# Pipeline 2 — Entity-Risk-Score Model

Same baseline features as Pipeline 1, **plus** entity-linkage features:
`card1_fraud_rate`, `card1_txn_count`, `addr1_fraud_rate`,
`email_domain_fraud_rate`, `entity_risk_score`. Same model family (LightGBM)
as P1, on the same split, so the AUC delta is a clean comparison.

**Leakage-safe recipe (applied exactly):**
1. The time-based split is fixed already (notebook 00) and reused as-is.
2. Entity fraud rate / count are grouped-by on **training rows only**
   (`features.fit_entity_lookups`).
3. Those lookups are stored, then mapped onto train/test rows by value
   (`features.apply_entity_lookups`) -- unseen entities fall back to the
   training-set global fraud rate, never 0, never a test-set rate.
4. The lookups are never refit after this point.

A 5-row leakage spot-check closes out this notebook, per the brief's
error-proofing checklist.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config as cfg, data, features, models, evaluate

df = data.load_raw()
full = data.attach_split(df)
train_df = full[full["split"] == "train"].copy()
test_df = full[full["split"] == "test"].copy()
y_train, y_test = train_df[cfg.TARGET_COL], test_df[cfg.TARGET_COL]

In [2]:
# Fit lookups on TRAIN ONLY, then apply to both splits.
lookups = features.fit_entity_lookups(train_df)
train_e = features.apply_entity_lookups(train_df, lookups)
test_e = features.apply_entity_lookups(test_df, lookups)

assert (train_e[cfg.ID_COL].values == train_df[cfg.ID_COL].values).all()
assert (test_e[cfg.ID_COL].values == test_df[cfg.ID_COL].values).all()

print("Global training fraud rate (fallback for unseen entities):", round(lookups["global_rate"], 4))
train_e[cfg.ENTITY_RISK_FEATURES].describe()

Global training fraud rate (fallback for unseen entities): 0.0351


,card1_fraud_rate,card1_txn_count,addr1_fraud_rate,email_domain_fraud_rate,entity_risk_score
count,472432.000000,472432.000000,472432.000000,472432.000000,472432.000000
mean,0.035135,2036.637057,0.026162,0.035962,0.032420
std,0.065875,2984.747541,0.014210,0.014790,0.024325
min,0.000000,1.000000,0.000000,0.000000,0.000000
25%,0.002016,105.000000,0.020000,0.022930,0.020976
50%,0.016225,728.000000,0.024515,0.035135,0.026301
75%,0.038878,2536.000000,0.030652,0.044003,0.034109
max,1.000000,12205.000000,0.666667,0.462687,0.527723


In [3]:
X_train, encoder = features.prepare_baseline_matrix(
    train_e, extra_numeric_cols=cfg.ENTITY_RISK_FEATURES, fit_encoder=True
)
X_test, _ = features.prepare_baseline_matrix(
    test_e, extra_numeric_cols=cfg.ENTITY_RISK_FEATURES, encoder=encoder, fit_encoder=False
)
print(X_train.shape, X_test.shape)

lgbm2 = models.train_lgbm(X_train, y_train)
p2_scores = models.predict_lgbm(lgbm2, X_test)
p2_result = evaluate.evaluate_scores(y_test, p2_scores)
print("Pipeline 2 (baseline + entity-risk):", p2_result)

(472432, 40) (118108, 40)


Pipeline 2 (baseline + entity-risk): {'auc': np.float64(0.8604586252211813), 'pr_auc': np.float64(0.4509042548433899), 'threshold': 0.5, 'precision': np.float64(0.24699076234020714), 'recall': np.float64(0.6513287401574803), 'f1': np.float64(0.35816250591976184), 'tp': 2647, 'fp': 8070, 'fn': 1417, 'tn': 105974}


## Live-Demo Step 3 (Full Project Plan, Section 9.6)

"Run the entity-feature join + P2 model, print the new AUC next to P1's, let
the delta speak for itself." P1's saved score file (from notebook 02) is
loaded here just for this side-by-side printout -- the authoritative,
single-source-of-truth comparison across ALL pipelines still lives in
notebook 06.

In [4]:
p1_path = cfg.FLAGS_DIR / "p1_baseline_scores.csv"
if p1_path.exists():
    p1 = pd.read_csv(p1_path)
    p1_scores = p1.set_index(cfg.ID_COL).loc[test_e[cfg.ID_COL], "p1_lgbm_score"].values
    p1_result = evaluate.evaluate_scores(y_test, p1_scores)
    auc_delta = p2_result["auc"] - p1_result["auc"]
    # PR-AUC delta reported side-by-side with the ROC-AUC delta -- with a ~3.5%
    # fraud base rate, ROC-AUC alone can look flat/negative while PR-AUC (driven
    # only by the fraud class) tells a different story, or vice versa. Both are
    # first-class findings for this track, not just ROC-AUC with PR-AUC as a footnote.
    pr_auc_delta = p2_result["pr_auc"] - p1_result["pr_auc"]
    print("=" * 60)
    print(f"  P1 (baseline, no entity signals)       ROC-AUC = {p1_result['auc']:.4f}   PR-AUC = {p1_result['pr_auc']:.4f}")
    print(f"  P2 (baseline + entity-risk-score)      ROC-AUC = {p2_result['auc']:.4f}   PR-AUC = {p2_result['pr_auc']:.4f}")
    print(f"  DELTA (P2 - P1)                        ROC-AUC = {auc_delta:+.4f}   PR-AUC = {pr_auc_delta:+.4f}")
    print("=" * 60)
    if (auc_delta > 0) != (pr_auc_delta > 0):
        print(
            "  NOTE: ROC-AUC and PR-AUC disagree in direction here -- the entity "
            "layer's effect on the minority fraud class is mixed, not a clean "
            "win or loss. Worth flagging explicitly, not averaging away."
        )
else:
    print("Run notebook 02 first to generate p1_baseline_scores.csv for the side-by-side AUC/PR-AUC printout.")

  P1 (baseline, no entity signals)       ROC-AUC = 0.8778   PR-AUC = 0.4453
  P2 (baseline + entity-risk-score)      ROC-AUC = 0.8605   PR-AUC = 0.4509
  DELTA (P2 - P1)                        ROC-AUC = -0.0173   PR-AUC = +0.0056
  NOTE: ROC-AUC and PR-AUC disagree in direction here -- the entity layer's effect on the minority fraud class is mixed, not a clean win or loss. Worth flagging explicitly, not averaging away.


In [5]:
# How much do the entity-risk features matter to this specific model, relative to the baseline features?
importances = pd.Series(lgbm2.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(15))
print("\nTotal importance share held by entity-risk features:",
      round(importances[cfg.ENTITY_RISK_FEATURES].sum() / importances.sum(), 3))

card1_fraud_rate           2391
entity_risk_score          2250
txn_day                    2021
card1_txn_count            2003
TransactionAmt             1846
addr1_fraud_rate           1722
D1                         1318
C13                        1277
D15                        1111
email_domain_fraud_rate    1082
C1                          982
D4                          904
D10                         878
C2                          785
C14                         727
dtype: int32

Total importance share held by entity-risk features: 0.381


## Leakage spot-check (required by the error-proofing checklist)

Pick 5 random **test-split** rows and manually recompute their `card1_fraud_rate`
from training rows only, independent of the pipeline code above. If the stored
value doesn't match, or if it was computed using any test-row label, that's a
leak.

In [6]:
sample = test_e.sample(5, random_state=cfg.RANDOM_STATE)
all_match = True
for _, row in sample.iterrows():
    train_only_rows = train_df.loc[train_df["card1"] == row["card1"], cfg.TARGET_COL]
    manual_rate = train_only_rows.mean() if len(train_only_rows) else lookups["global_rate"]
    match = np.isclose(row["card1_fraud_rate"], manual_rate)
    all_match &= match
    print(f"TransactionID={row[cfg.ID_COL]} card1={row['card1']} "
          f"stored={row['card1_fraud_rate']:.4f} manual_train_only={manual_rate:.4f} match={match}")

assert all_match, "Leakage spot-check FAILED -- stored entity rate does not match a train-only recomputation."
print("\nLeakage spot-check PASSED: all 5 sampled test-row entity rates match a train-only recomputation.")

TransactionID=3468994 card1=12778 stored=0.1495 manual_train_only=0.1495 match=True
TransactionID=3522291 card1=7826 stored=0.0212 manual_train_only=0.0212 match=True
TransactionID=3465762 card1=17399 stored=0.0080 manual_train_only=0.0080 match=True
TransactionID=3465720 card1=4109 stored=0.0000 manual_train_only=0.0000 match=True
TransactionID=3503231 card1=11960 stored=0.0000 manual_train_only=0.0000 match=True

Leakage spot-check PASSED: all 5 sampled test-row entity rates match a train-only recomputation.


In [7]:
out = test_df[[cfg.ID_COL]].copy()
out["p2_lgbm_score"] = p2_scores
out_path = cfg.FLAGS_DIR / "p2_entity_risk_scores.csv"
out.to_csv(out_path, index=False)
print("Saved:", out_path, out.shape)

Saved: /Users/derr/Documents/Random/SP2718F/Group4-TrackD/analysis/outputs/flags/p2_entity_risk_scores.csv (118108, 2)
